# Lotofácil — Histórico completo e sugestão estatística para o próximo concurso

Este notebook usa as funções de `lotofacil_analise.py` (mesma pasta do repositório) para:

1. Baixar automaticamente **todo o histórico** de concursos da Lotofácil (com cache local incremental em `lotofacil_historico.csv`);
2. Fazer uma análise estatística do histórico (frequência, atraso, pares, paridade, soma, repetição, distribuição por coluna da cartela);
3. Gerar uma sugestão de jogo para o próximo concurso.

> **Importante:** a Lotofácil é um sorteio aleatório — cada concurso é um evento independente e concursos passados **não** influenciam o próximo sorteio. Nenhum método estatístico garante ou comprovadamente aumenta a chance de acerto. Este notebook tem fins exploratórios/educacionais. Jogue com responsabilidade.


In [ ]:
import sys
from pathlib import Path

# garante que o Python encontre lotofacil_analise.py mesmo se o notebook
# for aberto de outro diretório de trabalho
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from lotofacil_analise import (
    atualizar_historico,
    historico_ordenado,
    analisar,
    gerar_sugestoes,
    montar_relatorio,
    COLUNAS_CARTELA,
)

# paleta sequencial (uma única cor, do claro ao escuro) usada nos gráficos abaixo
AZUL_SEQUENCIAL = LinearSegmentedColormap.from_list(
    "azul_sequencial", ["#cde2fb", "#6da7ec", "#2a78d6", "#104281"]
)
COR_BARRA_UNICA = "#2a78d6"
COR_GRADE = "#e1e0d9"
COR_EIXO = "#c3c2b7"
COR_TEXTO_SECUNDARIO = "#52514e"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": COR_EIXO,
    "axes.grid": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "text.color": "#0b0b0b",
    "axes.labelcolor": COR_TEXTO_SECUNDARIO,
    "xtick.color": COR_TEXTO_SECUNDARIO,
    "ytick.color": COR_TEXTO_SECUNDARIO,
})


## 1. Baixar/atualizar o histórico

Na primeira execução baixa todos os concursos (pode levar alguns minutos); nas próximas, baixa só os concursos novos.

In [ ]:
historico = atualizar_historico()  # use atualizar_historico(forcar=True) para reforçar o download completo
lista = historico_ordenado(historico)

print(f"Total de concursos em cache: {len(lista)}")
print(f"Do concurso {lista[0][0]} ao {lista[-1][0]} (último sorteio em {lista[-1][1]})")


## 2. Parâmetros da análise

In [ ]:
JANELA_RECENTE = 25     # quantos concursos recentes entram na análise de tendência
QUANTIDADE_APOSTA = 16  # a Lotofácil aceita apostas de 15 a 20 números

resultado = analisar(lista, janela_recente=JANELA_RECENTE)
resultado["total_concursos"], resultado["soma_media"], resultado["soma_desvio"]


## 3. Números mais e menos frequentes

In [ ]:
df_freq = pd.DataFrame({
    "dezena": list(range(1, 26)),
    "frequencia_total": [resultado["freq_total"].get(d, 0) for d in range(1, 26)],
    "frequencia_recente": [resultado["freq_recente"].get(d, 0) for d in range(1, 26)],
    "atraso": [resultado["atraso"].get(d, 0) for d in range(1, 26)],
})
df_freq["pct_total"] = (100 * df_freq["frequencia_total"] / resultado["total_concursos"]).round(1)

df_freq.sort_values("frequencia_total", ascending=False).head(10).reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ordem = df_freq.sort_values("dezena")
cores = AZUL_SEQUENCIAL(
    (ordem["frequencia_total"] - ordem["frequencia_total"].min())
    / (ordem["frequencia_total"].max() - ordem["frequencia_total"].min() + 1e-9)
)
ax.bar(ordem["dezena"], ordem["frequencia_total"], color=cores)
ax.set_xticks(range(1, 26))
ax.set_xlabel("Dezena")
ax.set_ylabel("Vezes sorteada")
ax.set_title("Frequência histórica de cada dezena")
plt.tight_layout()
plt.show()


## 4. Atraso — dezenas que não saem há mais tempo

In [ ]:
df_freq.sort_values("atraso", ascending=False).head(10).reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ordem_atraso = df_freq.sort_values("atraso", ascending=False)
ax.bar(ordem_atraso["dezena"].astype(str), ordem_atraso["atraso"], color=COR_BARRA_UNICA)
ax.set_xlabel("Dezena")
ax.set_ylabel("Concursos sem sair")
ax.set_title(f"Atraso de cada dezena (ordenado do mais ao menos atrasado)")
plt.tight_layout()
plt.show()


## 5. Pares de números que mais saem juntos

In [ ]:
df_pares = pd.DataFrame(
    [(a, b, qtd) for (a, b), qtd in resultado["pares"].most_common(15)],
    columns=["dezena_a", "dezena_b", "vezes_juntas"],
)
df_pares


## 6. Paridade e soma das 15 dezenas sorteadas

In [ ]:
df_paridade = pd.DataFrame(
    sorted(resultado["contagem_pares_impares"].items()),
    columns=["quantidade_de_pares", "concursos"],
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_paridade["quantidade_de_pares"].astype(str), df_paridade["concursos"], color=COR_BARRA_UNICA)
ax.set_xlabel("Números pares no jogo (de 15)")
ax.set_ylabel("Concursos")
ax.set_title("Distribuição de pares x ímpares por concurso")
plt.tight_layout()
plt.show()

print(f"Soma média histórica: {resultado['soma_media']:.1f}  |  Desvio padrão: {resultado['soma_desvio']:.1f}")
print(f"Repetição média em relação ao concurso anterior: {resultado['repeticao_media']:.1f} dezenas")


## 7. Distribuição por coluna da cartela oficial

In [ ]:
df_coluna = pd.DataFrame(
    sorted(resultado["freq_coluna"].items()),
    columns=["coluna", "ocorrencias"],
)
df_coluna["dezenas"] = df_coluna["coluna"].map(lambda c: COLUNAS_CARTELA[c])
df_coluna


## 8. Sugestão para o próximo concurso

Combina frequência histórica, tendência recente e atraso de cada dezena; o núcleo de 15 dezenas é ajustado para manter a soma dentro da faixa mais comum do histórico (média ± 1 desvio padrão), e — como `QUANTIDADE_APOSTA` está em 16 — é completado com a próxima dezena mais bem colocada no ranking.

In [ ]:
sugestao, pontuacao, alternativas = gerar_sugestoes(resultado, quantidade=QUANTIDADE_APOSTA)

print(f"Sugestão principal ({QUANTIDADE_APOSTA} números):")
print(" - ".join(f"{d:02d}" for d in sugestao))
n_pares_sug = sum(1 for d in sugestao if d % 2 == 0)
print(f"Soma: {sum(sugestao)}  |  Pares: {n_pares_sug}  |  Ímpares: {QUANTIDADE_APOSTA - n_pares_sug}")

print("\nEstratégias alternativas para comparação:")
for nome, jogo in alternativas.items():
    print(f"  {nome}: {' - '.join(f'{d:02d}' for d in jogo)}  (soma {sum(jogo)})")


## 9. Relatório completo (texto)

In [ ]:
relatorio = montar_relatorio(resultado, lista, quantidade=QUANTIDADE_APOSTA)
print(relatorio)

# opcional: salvar em arquivo
# Path("relatorio_lotofacil.txt").write_text(relatorio, encoding="utf-8")


---

**Aviso:** a Lotofácil é um sorteio aleatório. O resultado de concursos passados não influencia o próximo sorteio, e nenhuma análise estatística garante ou comprovadamente aumenta a chance de acerto. Use esta ferramenta apenas para fins exploratórios/educacionais e jogue com responsabilidade.